### Coding Task: EK Model

Submission by Ruhani Walia

#### Table of Contents
<a id="top"></a>

- [Part ii](#two) 
- [Part iii](#three)  
- [Part iv](#four) 
- [Part v](#five) 

In [59]:
# 0. SETUP NOTEBOOK ENVIRONMENT
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm

# Set the input path to the directory containing the input files
input_path = Path(r"C:\Users\ruhan\Desktop\UChicago\Applied Economics Incubator App\Input")    

trade_data = input_path / "bilateral_trade_country.csv"
country_names = input_path / "country_list.csv"

# Set the output path
output_path = Path(r"C:\Users\ruhan\Desktop\UChicago\Applied Economics Incubator App\Output")

# The assignment tells us to calibrate the trade elasticity to 4; I define it here
THETA = 4.0

##### Part ii: Calibrating the baseline of the model
<a id="two"></a>
[Back to Top](#top)

This section reads in the trade data and builds three objects:

- **X0**: the \(N x N\) matrix of bilateral trade flows, \(X_0[n,i]\)
- **y0**: the vector of income-to-spending ratios, \(y_0[n]\)
- **theta**: the trade elasticity (fixed at 4)

**Note on notation and conventions**

- i = origin / exporter
- n = destination / importer
- X_0[n,i] = value of goods that country n **buys from** country i

This means that **rows are destinations n** and **columns are origins i**.

Diagonal entries \(X_0[n,n]\) are **domestic sales** (country \(n\) buying from itself).

In [60]:
# 1. LOAD THE RAW DATA

# The trade file is read in with one row per (i, n) pair 
# country_org  = origin country i (the seller)
# country_dest = destination country n (the buyer)
# trade_flow2014 = value of goods flowing from i to n (X-tilde_ni)

trade_data = pd.read_csv(input_path / "bilateral_trade_country.csv")
country_org = trade_data["country_org"].str.strip().tolist()
country_dest = trade_data["country_dest"].str.strip().tolist()

# The country list is just the 41 country codes
# Used to calibrate order of rows and columns of the matrix, so that row k and column k refer to the same country
# This is so that the diagonal of the matrix represents domestic trade flows

country_names = pd.read_csv(input_path / "country_list.csv")
countries = country_names["country"].str.strip().tolist()
N = len(countries)  # number of countries

In [61]:
# 2. DATA COMPLETENESS CHECKS

# 2.1. Check that the trade data and country list are consistent
# i.e., every country in the trade file should be in the country list, and vice versa

trade_countries = set(country_org) | set(country_dest)
country_set = set(countries)
# print(trade_countries)
# print(len(trade_countries))

# Countries appearing in trade data but not in country list
extra_in_trade = trade_countries - country_set
# print(country_set)
# print(len(country_set))

# Countries in country list but not appearing in trade data
missing_from_trade = country_set - trade_countries

print("Countries in trade data but not country list:", extra_in_trade)
print("Countries in country list but not trade data:", missing_from_trade)

assert trade_countries == country_set, "Trade data and country list are inconsistent."

print("Check passed: trade data and country list contain the same countries.")


Countries in trade data but not country list: set()
Countries in country list but not trade data: set()
Check passed: trade data and country list contain the same countries.


In [62]:
# 2.2 Check that there are no duplicate (i, n) pairs in the trade data
duplicates = trade_data.duplicated(
    subset=["country_org", "country_dest"],
    keep=False
)

print("Number of rows with duplicate (i, n) pairs:", duplicates.sum())

assert not duplicates.any(), "Duplicate (i, n) pairs found."

print("Check passed: no duplicate (i, n) pairs.")

Number of rows with duplicate (i, n) pairs: 0
Check passed: no duplicate (i, n) pairs.


In [63]:
# 2.3 Check that the trade data is a complete N x N panel 
# i.e., every pair is observed, including where i == n
expected_pairs = N ** 2
actual_pairs = len(trade_data)

print("Expected number of (i, n) pairs:", expected_pairs)
print("Actual number of (i, n) pairs:", actual_pairs)

assert actual_pairs == expected_pairs, "Trade data is not a complete N x N panel."

print("Check passed: trade data contains all N x N country pairs.")

Expected number of (i, n) pairs: 1681
Actual number of (i, n) pairs: 1681
Check passed: trade data contains all N x N country pairs.


In [64]:
# 2.4 Check that there are no missing or negative trade values
assert trade_data["trade_flow2014"].notna().all(), \
    "Missing trade values found."

assert (trade_data["trade_flow2014"] >= 0).all(), \
    "Negative trade values found."

print("Check passed: no missing or negative trade values.")

Check passed: no missing or negative trade values.


In [66]:
# 3.0 CALIBRATE BILATERAL TRADE MATRIX, X0_ni
# X˜ni = from country i to country n 
# country i = export
# country n = import

# Want to create an NxN matrix such that:
#   country_dest (n) is the row index
#   country_org (i) is the column index
#   values = trade_flow2014 (value of trade flow X˜ni = from country i to country n)
# This way, entry X0[n, i] corresponds to "n's purchases from i"

# To create a matrix with this design, pivot the input data
# .reindex ensures that the rows and columns are in the same order as the country list
# This allows the diagonal to represent domestic trade flows
X0_df = (
    trade_data.pivot(index="country_dest", columns="country_org", values="trade_flow2014")
    .reindex(index=countries, columns=countries)
)

print(X0_df)

country_org              AUS           AUT           BEL          BGR  \
country_dest                                                            
AUS          2,437,913.45470   1,228.68941   3,426.57660     36.91224   
AUT                197.21331 598,635.34171   5,488.87907    921.13228   
BEL                738.51643   3,154.92848 836,791.62661  1,672.50389   
BGR                192.05977   1,035.14091     789.97946 91,174.20634   
BRA              1,952.17615   1,373.75138   2,519.88231     37.51372   
CAN              1,806.54042   1,906.55886   2,078.88630     85.94858   
CHE              1,086.12311  10,359.91505  14,056.90491    224.80094   
CHN             76,645.19547   6,561.15501   9,733.52592    816.40805   
CZE                143.10687   5,736.82033   3,672.12459    420.99678   
DEU              1,601.52638  61,746.13170  74,055.66072  2,877.75569   
DNK                328.85991   1,015.74219   3,874.78169    186.86085   
ESP                673.83238   2,903.51360  10,046.

In [67]:
# For use later, convert the DataFrame to a NumPy array
# This will allow indexing by integer position where X0_ni[n, i] corresponds to "n's purchases from i"
X0_ni = X0_df.to_numpy()

# Create dictionary for each country's position 
idx = {c: k for k, c in enumerate(countries)}

# axis=0 is sum DOWN rows --> column totals
# axis=1 is sum ACROSS columns --> row totals

In [68]:
# e.g., to find the position of US and CA:
# idx['CAN']  gives that Canada is at position 5
# idx['USA']  gives that United States is at position 40
# So X0[40, 5] = US's purchases from CA
X0_ni[40, 5]

np.float64(350482.1343441567)

In [69]:
# 4.0 COMPUTE VECTOR OF INCOME-TO-SPENDING RATIOS

# 4.1 COMPUTE TOTAL SPENDING FOR EACH COUNTRY
# This is total spending by country n on goods from all origins (including domestic goods)
# X0_n = the sum over origins i of X0_ni[n, i] i.e., sum of each row n in the matrix X0_ni
X0_n = X0_ni.sum(axis=1)

# Quick check
# Directly from raw trade data
# canada_raw = trade_data.loc[
#     trade_data["country_dest"] == "CAN",
#     "trade_flow2014"
# ].sum()

# # From constructed matrix
# canada_matrix = X0_n[idx["CAN"]]

# print("Raw data:", canada_raw)
# print("Matrix:", canada_matrix)
# print("Difference:", canada_raw - canada_matrix)

In [70]:
# 4.2 COMPUTE TOTAL SALES FOR EACH COUNTRY
# This is the sum of all exports for country n
# Y0_n = the sum over destinations n of X0_ni[n, i] i.e., sum of each col i in the matrix X0_ni

Y0_n = X0_ni.sum(axis=0)

In [71]:
# 4.3 COMPUTE INCOME-TO-SPENDING RATIOS y0_n

# Given definition:  y0_n = ( sum_i X0_in ) / ( sum_i X0_ni )
#   numerator   = sum_i X0[i, n] = column n sum = total SALES of n = w0_n L0_n = Y0_n (by market clearing condition)
#   denominator = sum_i X0[n, i] = row n sum    = total SPENDING of n = X0_n

# So, y0_n = Y0_n / X0_n = (labour income / total spending)
#   y0_n < 1 : n spends more than it earns  -> trade DEFICIT (net transfer D_n > 0)
#   y0_n > 1 : n earns more than it spends  -> trade SURPLUS (net transfer D_n < 0)
#   y0_n = 1 : trade is balanced

assert np.all(X0_n > 0), "At least one country has zero total spending."

y0_n = Y0_n / X0_n

assert np.isfinite(y0_n).all(), "Non-finite income-to-spending ratio found."

In [72]:
# Check that the computed y0_n values are consistent
y0_check = pd.DataFrame({
    "country": countries,
    "total_sales": Y0_n,
    "total_spending": X0_n,
    "y0_n": y0_n
})

y0_check

,country,total_sales,total_spending,y0_n
0,AUS,"2,725,247.38608","2,726,903.42542",0.99939
1,AUT,"809,693.79453","793,432.60270",1.02049
2,BEL,"1,322,905.97082","1,279,536.37772",1.03389
3,BGR,"122,872.50677","125,719.92096",0.97735
4,BRA,"4,103,504.28490","4,150,582.29601",0.98866
5,CAN,"3,256,460.74983","3,237,575.88852",1.00583
6,CHE,"1,401,663.11614","1,314,600.56617",1.06623
7,CHN,"31,745,423.56244","31,162,795.62772",1.01870
8,CZE,"492,779.00461","473,407.72111",1.04092
9,DEU,"7,070,763.12631","6,680,362.23433",1.05844


##### Part iii: Productivity boost counterfactual
<a id="three"></a>
[Back to Top](#top)

This section solves for the change in every country's wage (relative to the US wage) after a 10% productivity increase in China.
That is, T_hat_China = 1.1. Excess labour demand Z_i is defined for each country i, as a function of the vector of wage changes w_hat. The equilibrium wage changes are the ones where Z_i = 0 for every country (labour demand = labour supply everywhere). 
This is found by starting from a guess and nudging wages up where Z_i > 0 (excess demand) and down where Z_i < 0.

In [73]:
# 1.0 COMPUTE TRADE SHARES pi0
# pi0[n, i] = X0[n, i] / X0_n = fraction of n's spending that goes to goods purchased from i

pi0 = X0_ni / X0_n[:, None]

# # Manual check: Canada's spending share on US goods
# can_idx = idx["CAN"]
# usa_idx = idx["USA"]

# # Bilateral trade flow: Canada buys from USA
# canada_from_usa = X0_ni[can_idx, usa_idx]

# # Canada's total spending
# canada_spending = X0_n[can_idx]

# # Calculate the share manually
# manual_share = canada_from_usa / canada_spending

# # Compare to the corresponding entry in pi0
# pi0_share = pi0[can_idx, usa_idx]

# print("Canada purchases from USA:", canada_from_usa)
# print("Canada total spending:", canada_spending)
# print("Manual share:", manual_share)
# print("pi0 share:", pi0_share)
# print("Difference:", manual_share - pi0_share)

In [74]:
# Some sanity checks prior to counterfactual analysis

# Trade shares for each destination should sum to 1
assert np.allclose(pi0.sum(axis=1), 1.0), \
    "Trade shares do not sum to 1 within each row"

# Income-to-spending ratios should be strictly positive
assert (y0_n > 0).all(), \
    "Non-positive income-to-spending ratio"

# International transfers should satisfy the adding-up condition
transfers = (1.0 - y0_n) * X0_n

assert np.isclose(
    transfers.sum(),
    0.0,
    atol=1e-6 * X0_n.sum()
), "Transfers do not sum to zero"

print("All calibration checks passed.")

All calibration checks passed.


In [79]:
# 2.0 DEFINE EXCESS LABOUR DEMAND FUNCTION

# 2.1 DEFINE HELPER FUNCTION TO COMPUTE EXCESS LABOUR DEMAND
def model_objects(w_hat, T_hat, d_hat, L_hat, D_hat, transfer_idx):
    """
    Given a guess for the wage changes w_hat (vector of length N) and the known shocks, compute the intermediate objects of the 
    hat-algebra system.

    w_hat is a guess for the wage changes for each country. It is the unknown we are solving for and the solver revises 
    it until the excess labour demand is zero for all countries.

    Arguments
    ---------
    w_hat        : (N,)   guess for wage changes (ratio new/old); w_hat[US] = 1
    T_hat        : (N,)   productivity shocks (ratio new/old)
    d_hat        : (N, N) trade cost shocks, d_hat[n, i] for shipping i --> n
    L_hat        : (N,)   labour force changes
    D_hat        : (N,)   transfer changes
    transfer_idx : int    position of the country whose wage is the numeraire of transfers
                          h(w) = w_{transfer_idx} i.e., for this section, it is the US

    Returns
    -------
    Phi_hat : (N,)   change in each country's "market favourability/price environment"
    pi_hat  : (N, N) change in trade shares, pi_hat[n, i]
    X_hat   : (N,)   change in each country's total spending
    """
    # 1. the "cost" of buying from each origin o, in hat form
    # For destination n and origin o, the term is  T_hat_o * (w_hat_o * d_hat_no)^(-theta)
    # A higher wage (or higher trade cost) in origin o makes it a worse place to
    # buy from, so this term falls as w_hat_o rises (theta > 0 and the exponent is -theta)
    cost_term = T_hat[None, :] * (w_hat[None, :] * d_hat) ** (-THETA)

    # 2. Phi hat, the change in market favourability/price environment
    # Phi_hat_n = sum_o pi0[n, o] * T_hat_o * (w_hat_o * d_hat_no)^(-theta)
    Phi_hat = (pi0 * cost_term).sum(axis=1)

    # 3. pi_hat, the change in trade shares
    # pi_hat[n, i] = T_hat_i (w_hat_i d_hat_ni)^(-theta) / Phi_hat_n
    pi_hat = cost_term / Phi_hat[:, None]

    # 4. X_hat, the change in total spending
    # X_hat_n = y0_n * w_hat_n * L_hat_n + (1 - y0_n) * [h(w0 w_hat)/h(w0)] * D_hat_n
    h_ratio = w_hat[transfer_idx]
    X_hat = y0_n * w_hat * L_hat + (1.0 - y0_n) * h_ratio * D_hat

    return Phi_hat, pi_hat, X_hat

In [ ]:
# 2.2 DEFINE FUNCTION TO COMPUTE EXCESS LABOUR DEMAND

def excess_demand(w_hat, T_hat, d_hat, L_hat, D_hat, transfer_idx):
    """
    The excess labour demand Z_i(w_hat) for every country i.
    Returns a vector of length N.
    """
    Phi_hat, pi_hat, X_hat = model_objects(w_hat, T_hat, d_hat, L_hat, D_hat, transfer_idx)

    # Share of country i's baseline sales that go to destination n:
    #     X0_ni[n, i] / sum_d X0_ni[d, i] = X0_ni[n, i] / Y0_n[i]
    sales_share = X0_ni / Y0_n[None, :]

    # New demand for i's labour, as a ratio to i's baseline income:
    #     sum_n sales_share[n, i] * pi_hat[n, i] * X_hat[n]
    # Summing over n (axis=0, DOWN each column) gives one number per origin i.
    demand = (sales_share * pi_hat * X_hat[:, None]).sum(axis=0)

    # Labour supply change: w_hat_i * L_hat_i (what i's labour income would be)
    supply = w_hat * L_hat

    # Z_i > 0 : the world wants more of i's labour than i supplies -> wage should rise
    # Z_i < 0 : the reverse                                       -> wage should fall
    return demand - supply

In [ ]:
# 3.0 DEFINE THE ITERATIVE ALGORITHM TO SOLVE FOR WAGE CHANGES

def solve_wages(T_hat, d_hat, L_hat, D_hat, transfer_idx, numeraire_idx,
                kappa=0.5, tol=1e-10, max_iter=100_000, verbose=True):
    """
    Iterate  w_hat_i(b+1) = w_hat_i(b) + kappa * Z_i(w_hat(b))  for all i except
    the numeraire country, until max_i |Z_i| < tol

    numeraire_idx : position of the country whose wage is held fixed at 1 (the US)
                    Its wage is not updated and it is excluded from the convergence check
    """
    # Initial guess: no change in any wage
    w_hat = np.ones(N)

    # Boolean mask for "every country except the numeraire"
    free = np.ones(N, dtype=bool)
    free[numeraire_idx] = False

    history = []  # store max|Z| at each step to check it is shrinking

    for b in range(max_iter):
        # Compute excess labour demand at the current guess
        Z = excess_demand(w_hat, T_hat, d_hat, L_hat, D_hat, transfer_idx)

        # Largest absolute excess demand across the non-numeraire countries
        max_err = np.abs(Z[free]).max()
        history.append(max_err)

        if verbose and (b % 100 == 0):
            print(f"  iteration {b:6d}   max|Z| = {max_err:.3e}")

        # Stopping rule: converged
        if max_err < tol:
            if verbose:
                print(f"Converged after {b} iterations (max|Z| = {max_err:.3e} < tol = {tol}).")
            return w_hat, np.array(history)

        # Raise wages where demand exceeds supply
        # Only the non-numeraire countries move; the US stays at exactly 1
        w_hat[free] = w_hat[free] + kappa * Z[free]

    raise RuntimeError(
        f"Did not converge in {max_iter} iterations (last max|Z| = {history[-1]:.3e}). "
        "Try a smaller kappa."
    )

In [ ]:
# 4.0 SET UP AND RUN THE COUNTERFACTUAL

if __name__ == "__main__":
    US, CHN = idx["USA"], idx["CHN"]

    # Everything shock is "no change" (= 1 in hat form) except Chinese productivity
    T_hat = np.ones(N)
    T_hat[CHN] = 1.1              # 10% increase in Chinese productivity
    d_hat = np.ones((N, N))       # no change in trade costs
    L_hat = np.ones(N)            # no change in labour forces
    D_hat = np.ones(N)            # no change in transfers

    # Check that with no shock wages do not move
    # If set every hat to 1 and w_hat = 1, then Z_i must be exactly 0 for all countries
    Z_null = excess_demand(np.ones(N), np.ones(N), d_hat, L_hat, D_hat, transfer_idx=US)
    print(f"Null-shock check: max|Z| at w_hat = 1, no shock = {np.abs(Z_null).max():.2e}  (should be ~0)")
    assert np.abs(Z_null).max() < 1e-10, "Baseline is not an equilibrium: check part ii / equation (2)"
    print()

    # transfers are denominated in US wages:  h(w) = w_US  ->  transfer_idx = US
    # the US wage is the numeraire (fixed at 1) ->           numeraire_idx = US
    print("Solving for wage changes after a 10% Chinese productivity increase...")
    w_hat, history = solve_wages(
        T_hat, d_hat, L_hat, D_hat,
        transfer_idx=US, numeraire_idx=US,
        kappa=0.5, tol=1e-10,
    )

    # Check that max error is falling over time
    print()
    print(f"max|Z| at start: {history[0]:.3e},  at end: {history[-1]:.3e}")
    print(f"Error fell monotonically: {bool(np.all(np.diff(history) <= 1e-15))}")

    # Solution checks
    Z_final = excess_demand(w_hat, T_hat, d_hat, L_hat, D_hat, transfer_idx=US)
    # Labour market clears everywhere except (possibly) the numeraire country
    # By Walras' law the US market clears too once all others do:
    print(f"Excess demand for the US (numeraire, not solved for directly): {Z_final[US]:.2e}")
    # The US wage is exactly 1 by construction
    assert w_hat[US] == 1.0

    # Results
    # w_hat_i is the ratio new/old wage relative to the US wage
    # (w_hat - 1) * 100 is the percentage change
    results = pd.DataFrame(
        {"w_hat": w_hat, "pct_change_wage": (w_hat - 1.0) * 100.0},
        index=countries,
    ).sort_values("w_hat", ascending=False)

    pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
    print()
    print("Wage change relative to the US wage (w_hat = new/old; US = 1 by construction):")
    print(results.to_string())

    
    # Save results
    Phi_hat, pi_hat, X_hat = model_objects(w_hat, T_hat, d_hat, L_hat, D_hat, transfer_idx=US)
    pi_hat_own = np.diag(pi_hat)   # pi_hat_ii: change in domestic trade share
    print()
    print("Saved for part iv: w_hat, Phi_hat, pi_hat, X_hat (and pi_hat_own = diag of pi_hat).")

Null-shock check: max|Z| at w_hat = 1, no shock = 1.11e-15  (should be ~0)

Solving for wage changes after a 10% Chinese productivity increase...
  iteration      0   max|Z| = 1.242e-02
  iteration    100   max|Z| = 1.619e-06
  iteration    200   max|Z| = 3.366e-09
Converged after 257 iterations (max|Z| = 9.964e-11 < tol = 1e-10).

max|Z| at start: 1.242e-02,  at end: 9.964e-11
Error fell monotonically: True
Excess demand for the US (numeraire, not solved for directly): -3.74e-10

Wage change relative to the US wage (w_hat = new/old; US = 1 by construction):
      w_hat  pct_change_wage
CHN 1.02177          2.17721
TWN 1.00058          0.05778
KOR 1.00037          0.03699
AUS 1.00036          0.03551
BRA 1.00017          0.01722
ROW 1.00016          0.01650
DEU 1.00016          0.01611
JPN 1.00015          0.01509
SVK 1.00014          0.01402
DNK 1.00014          0.01400
AUT 1.00014          0.01388
CHE 1.00013          0.01289
IDN 1.00012          0.01164
NOR 1.00011          0.01148


**Part iii results: 10% increase in Chinese productivity**

This section solved for the change in each country's wage relative to the US wage (the numeraire, ŵ_US = 1) using the iterative algorithm, starting from ŵ_i = 1 for all i ≠ US, with κ = 0.5 and tol = 10⁻¹⁰. The algorithm converged in 257 iterations, and the maximum excess labour demand fell at every step. As a check, with no shock the excess demand is zero (about 10⁻¹⁵), and the US labour market, which was not solved for directly, also clears (about −4 × 10⁻¹⁰).

Chinese wages rise by about 2.18% relative to the US. Changes for other countries are very small, mostly between 0 and +0.06%. The largest gains outside China are in Taiwan (+0.058%), Korea (+0.037%) and Australia (+0.036%), while Mexico is slightly negative (−0.003%).

##### Part iv: Real wages and real expenditure
<a id="four"></a>
[Back to Top](#top)

This section checks if wage changes from the previous section explain who gains from China's productivity boost and what might explain the differences across countries.

In [83]:
# 1.0 CALCULATE PRICE INDEX CHANGE, P_hat

# In order to determine real wages (w_hat / P_hat) and real expenditure (X_hat / P_hat) must compute
# P_hat = change in each country's price index (so can divide nominal changes by change in cost of living to get real changes)

# In this model, the price index of a country n follows from the CES price index and G_n(p) = 1 - exp(-Phi_n p^theta)
# Taking the ratio new/old to get the hat algebra form, the constant gamma cancels, leaving
#       P_hat_n = Phi_hat_n^(-1/theta)
# A high Phi_n indicates a favourable price environment and thus low prices (which is why phi is raised to a negative power)

P_hat = Phi_hat ** (-1.0 / THETA)

In [84]:
# 2.0 CALCULATE REAL WAGE AND REAL EXPENDITURE CHANGES
#   real_wage_hat > 1 : workers in that country can buy more after the shock
#   real_exp_hat  > 1 : total spending buys more goods after the shock
real_wage_hat = w_hat / P_hat
real_exp_hat = X_hat / P_hat

# Get real wage and real expenditure for each country, and report them in a table sorted by real wage change
report = pd.DataFrame(
    {
        "w_hat": w_hat,                                  # nominal wage change (from part iii)
        "P_hat": P_hat,                                  # price index change
        "real_wage_hat": real_wage_hat,                  # w_hat / P_hat
        "real_wage_pct": (real_wage_hat - 1.0) * 100.0,  # as a % change
        "real_exp_hat": real_exp_hat,                    # X_hat / P_hat
        "real_exp_pct": (real_exp_hat - 1.0) * 100.0,    # as a % change
    },
    index=countries,
).sort_values("real_wage_hat", ascending=False)

pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
print("Real wage and real expenditure changes (ratios new/old; US wage is the numeraire):")
print(report.to_string())

Real wage and real expenditure changes (ratios new/old; US wage is the numeraire):
      w_hat   P_hat  real_wage_hat  real_wage_pct  real_exp_hat  real_exp_pct
CHN 1.02177 0.99786        1.02396        2.39639       1.02437       2.43719
TWN 1.00058 1.00038        1.00020        0.01990       1.00024       0.02376
KOR 1.00037 1.00026        1.00011        0.01143       1.00013       0.01296
ROW 1.00016 1.00006        1.00010        0.01021       1.00009       0.00944
AUS 1.00036 1.00029        1.00007        0.00658       1.00007       0.00655
NLD 1.00007 1.00002        1.00005        0.00544       1.00006       0.00604
JPN 1.00015 1.00010        1.00005        0.00467       1.00005       0.00455
DEU 1.00016 1.00012        1.00004        0.00434       1.00005       0.00529
IDN 1.00012 1.00007        1.00004        0.00432       1.00004       0.00437
RUS 1.00006 1.00002        1.00004        0.00401       1.00004       0.00423
HUN 1.00010 1.00006        1.00004        0.00379       1.0

In [85]:
# 3.0 BUILD DATASET FOR REGRESSION ANALYSIS

# The assignment says to run the regressions on the sample of countries excluding China, so it is dropped here
CHN = idx["CHN"]
keep = np.ones(N, dtype=bool)
keep[CHN] = False

# Response variables
ln_real_wage = np.log(real_wage_hat)    # ln(w_hat / P_hat)
ln_real_exp = np.log(real_exp_hat)      # ln(X_hat / P_hat)

# Predictors
# (a) change in the domestic trade share: ln(pi_hat_ii).
#     np.diag(pi_hat) picks out pi_hat[i, i], each country's share of its own
#     spending that goes to its own goods. If this falls (< 0 in logs), the
#     country is buying relatively more from abroad
ln_pi_hat_own = np.log(np.diag(pi_hat))

# (b) change in market access: ln(Phi_hat_i). Positive means the country's
#     overall price environment improved (cheaper goods available to it)
ln_Phi_hat = np.log(Phi_hat)

# (c) initial spending share on Chinese goods: pi0_{i,China}. This is a
#     pre-shock value giving how exposed country i was to China before
#     anything changed. It is in levels, not logs and not a change
pi0_china = pi0[:, CHN]

data = pd.DataFrame(
    {
        "ln_real_wage": ln_real_wage,
        "ln_real_exp": ln_real_exp,
        "ln_pi_hat_own": ln_pi_hat_own,
        "ln_Phi_hat": ln_Phi_hat,
        "pi0_china": pi0_china,
    },
    index=countries,
).loc[keep]          # drop China

In [86]:
# 4.0 RUN THE SIX REGRESSIONS
# Run one regression per (response, predictor) pair: 2 outcomes x 3 regressors = 6
# Each is a simple regression  y = a + b * x + error  with a constant
outcomes = {"ln_real_wage": "ln(real wage)", "ln_real_exp": "ln(real expenditure)"}
regressors = {
    "ln_pi_hat_own": "ln(pi_hat_ii)  [change in domestic share]",
    "ln_Phi_hat": "ln(Phi_hat_i)  [change in market access]",
    "pi0_china": "pi0_iChina  [initial share on Chinese goods]",
}

rows = []
fits = {}
for y_col, y_label in outcomes.items():
    for x_col, x_label in regressors.items():
        X = sm.add_constant(data[x_col])          # adds the intercept column
        fit = sm.OLS(data[y_col], X).fit()        # ordinary least squares
        fits[(y_col, x_col)] = fit
        rows.append(
            {
                "outcome": y_label,
                "regressor": x_label,
                "slope": fit.params[x_col],
                "std_err": fit.bse[x_col],
                "t_stat": fit.tvalues[x_col],
                "R2": fit.rsquared,
                "N": int(fit.nobs),
            }
        )

reg_table = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_colwidth", 60)
print()
print("Regression results (one simple regression per row; China excluded):")
print(reg_table.to_string(index=False))


Regression results (one simple regression per row; China excluded):
             outcome                                    regressor   slope  std_err                  t_stat     R2  N
       ln(real wage)    ln(pi_hat_ii)  [change in domestic share] -0.2500   0.0000 -2,560,207,327,372.1333 1.0000 40
       ln(real wage)     ln(Phi_hat_i)  [change in market access] -0.0782   0.0130                 -6.0192 0.4881 40
       ln(real wage) pi0_iChina  [initial share on Chinese goods]  0.0037   0.0004                 10.2466 0.7343 40
ln(real expenditure)    ln(pi_hat_ii)  [change in domestic share] -0.2856   0.0060                -47.6064 0.9835 40
ln(real expenditure)     ln(Phi_hat_i)  [change in market access] -0.0934   0.0144                 -6.4849 0.5253 40
ln(real expenditure) pi0_iChina  [initial share on Chinese goods]  0.0040   0.0005                  8.4388 0.6521 40


In [ ]:
# # 5.0 CHECK IF REAL WAGE IS PINNED DOWN BY THE DOMESTIC SHARE
# # For every country other than China, T_hat = 1 and d_hat_ii = 1, so from the
# # pi_hat formula:   pi_hat_ii = w_hat_i^(-theta) / Phi_hat_i
# # Taking logs:      ln(pi_hat_ii) = -theta * ln(w_hat_i) - ln(Phi_hat_i)
# #
# # And the real wage is:  ln(w_hat_i / P_hat_i) = ln(w_hat_i) + (1/theta) ln(Phi_hat_i)
# # Substituting ln(Phi_hat_i) from above:
# #
# #       ln(real wage_i) = -(1/theta) * ln(pi_hat_ii)      (exactly, for i != China)
# #
# # So a regression of ln(real wage) on ln(pi_hat_ii) should have slope -1/theta =
# # -0.25, and R^2 = 1. This is a strong test that P_hat was computed correctly.
# check_fit = fits[("ln_real_wage", "ln_pi_hat_own")]
# print()
# print(f"Check: slope of ln(real wage) on ln(pi_hat_ii) = {check_fit.params['ln_pi_hat_own']:.6f}"
#       f"  (theory: {-1.0 / THETA:.6f}),  R^2 = {check_fit.rsquared:.6f}  (theory: 1)")


Check: slope of ln(real wage) on ln(pi_hat_ii) = -0.250000  (theory: -0.250000),  R^2 = 1.000000  (theory: 1)


**Part iv results**

Every country gains after China's productivity boost (T̂_China = 1.1, a 10% increase), and these regressions ask what explains the differences across the 40 non-China countries.

Change in domestic trade share: For these countries the model implies the exact relationship ln(real wage) = −(1/θ) ln(pi_ii), so the slope is −1/θ = −0.25. On average, a 1% increase in the domestic trade share is associated with a 0.25% decrease in the real wage; equivalently, countries experiencing larger proportional declines in their domestic trade shares experience larger real-wage gains. Because the real wage rises in every country, ln(pi_ii) is negative in every country: the domestic share fell everywhere. So the countries whose domestic share fell the most, meaning those that shifted the most spending toward foreign goods (driven by cheaper Chinese goods), gained the most.

Change in market access: The slope is −0.078. A higher phi means lower prices and a more favourable price environment, but the real wage also depends on the nominal wage. Because phi is measured relative to the US wage, it falls when a country's own wage rises. Countries with the largest wage gains therefore have the largest falls in phi and also the largest real wage gains, which produces the negative slope.

Initial spending share on Chinese goods: The slope is 0.0037. A country that spent 1 percentage point more of its budget on Chinese goods before the shock has a real wage about 0.0037% higher, because the fall in Chinese prices matters more the more of your spending it covers. This is the most direct economic result of the three.

Real expenditure shows the same pattern (slopes −0.286, −0.093 and +0.0040). Countries with larger declines in domestic trade shares and market access measures, and countries with greater initial exposure to Chinese goods, experience larger gains in real expenditure. The coefficients differ from those for real wages because expenditure responds not only through the price index and wage adjustment but also through the transfer/spending component of the model.

##### Part v: Same shock, different denomination 
<a id="five"></a>
[Back to Top](#top)

This section runs the same experiment but instead has that international transfers are measured in Chinese wages instead of US wages.
The work shows that it does matter which country's wage international transfers are measured in. 

That is, when the transfer unit (China wage) rises:
- Countries that receive transfers (deficit countries, y0 < 1) get more purchasing power, so they gain more
- Countries that pay transfers (surplus countries, y0 > 1) pay more, so they gain less, or even lose

In [90]:
# 1.0 SAVE RESULTS FROM iii

# Looking at total spending in hat form:
#
#     X_hat_n = y0_n * w_hat_n * L_hat_n  +  (1 - y0_n) * [h(w0 w_hat)/h(w0)] * D_hat_n
#                \_____ labour income ____/    \______________ transfers ____________/
#
# The transfer part of country n's spending is (1 - y0_n) * X0_n at baseline:
#     y0_n < 1  ->  n runs a deficit, RECEIVES transfers (D_n > 0)
#     y0_n > 1  ->  n runs a surplus, PAYS transfers    (D_n < 0)
#
# Transfers are fixed in units of "one country's wage". If that country's wage
# goes up, each unit of transfer is worth more:
#     - deficit countries (receivers) get a bigger transfer  -> spend more
#     - surplus countries (payers)   pay a bigger transfer   -> spend less
# In part iii the unit was the US wage (fixed at 1), so this effect was absent
# In part v the unit is the Chinese wage, which rises because of the shock, so
# the transfers become more valuable in US-wage terms

# Copy the first set of results:
w_hat_us = w_hat.copy()                       # nominal wage change, transfers in US wage
real_wage_hat_us = real_wage_hat.copy()       # real wage change,    transfers in US wage
real_exp_hat_us = real_exp_hat.copy()         # real expenditure,    transfers in US wage

In [ ]:
# 2.0 SOLVE MODEL WITH TRANSFERS IN CHINESE WAGES
# Only one argument changes: transfer_idx = CHN instead of US
#   transfer_idx = CHN : h(w) = w_China -> the transfer factor is w_hat[CHN]
#   numeraire_idx = US : the US wage is STILL the world numeraire (fixed at 1, not updated, excluded from the convergence check)
# The two indices are separate: "which wage do we hold at 1?" and "which wage are transfers measured in?"

# Sanity check first: with no shock nothing should move, whatever the numeraire
Z_null_v = excess_demand(np.ones(N), np.ones(N), d_hat, L_hat, D_hat, transfer_idx=CHN)
print(f"Null-shock check (transfers in Chinese wage): max|Z| = {np.abs(Z_null_v).max():.2e}  (should be ~0)")
assert np.abs(Z_null_v).max() < 1e-10

print("Solving with transfers denominated in the Chinese wage...")
w_hat_cn, history_cn = solve_wages(
    T_hat, d_hat, L_hat, D_hat,
    transfer_idx=CHN, numeraire_idx=US,
    kappa=0.5, tol=1e-10,
)

# Rebuild the ingredients at the new solution (same as part iv, new wages)
Phi_hat_cn, pi_hat_cn, X_hat_cn = model_objects(
    w_hat_cn, T_hat, d_hat, L_hat, D_hat, transfer_idx=CHN
)

# Price index change and real variables (same formulas as part iv)
P_hat_cn = Phi_hat_cn ** (-1.0 / THETA)
real_wage_hat_cn = w_hat_cn / P_hat_cn
real_exp_hat_cn = X_hat_cn / P_hat_cn

# Check errors fell
# print(f"max|Z| at start: {history_cn[0]:.3e}, at end: {history_cn[-1]:.3e}")
# print(f"Error fell monotonically: {bool(np.all(np.diff(history_cn) <= 1e-15))}")

Null-shock check (transfers in Chinese wage): max|Z| = 1.11e-15  (should be ~0)
Solving with transfers denominated in the Chinese wage...
  iteration      0   max|Z| = 1.242e-02
  iteration    100   max|Z| = 1.134e-06
  iteration    200   max|Z| = 1.866e-09
Converged after 246 iterations (max|Z| = 9.780e-11 < tol = 1e-10).


In [ ]:
# 3.0 COMPARE THE TWO SCENARIOS COUNTRY BY COUNTRY
# Everything below is in percent changes. "diff" = (Chinese-wage numeraire) minus (US-wage numeraire)
# So positive means the country does better when transfers are denominated in Chinese wages
comparison = pd.DataFrame(
    {
        "y0": y0_n,                                                    # the key data feature
        "transfer_share": 1.0 - y0_n,                                  # > 0 receiver, < 0 payer
        "real_wage_US": (real_wage_hat_us - 1) * 100,                  # part iii/iv
        "real_wage_CN": (real_wage_hat_cn - 1) * 100,                  # part v
        "real_wage_diff": (real_wage_hat_cn - real_wage_hat_us) * 100,
        "real_exp_US": (real_exp_hat_us - 1) * 100,
        "real_exp_CN": (real_exp_hat_cn - 1) * 100,
        "real_exp_diff": (real_exp_hat_cn - real_exp_hat_us) * 100,
    },
    index=countries,
).sort_values("y0")     # sorted from biggest deficit country to biggest surplus country

pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
print()
print("Real wage / real expenditure (% change), transfers in US wage vs Chinese wage.")
print("Sorted by y0: top = deficit countries (y0 < 1), bottom = surplus countries (y0 > 1)")
print(comparison.to_string())

print()
print(f"Chinese wage change (relative to US): {(w_hat_us[CHN]-1)*100:.4f}% with US-wage transfers,"
      f" {(w_hat_cn[CHN]-1)*100:.4f}% with Chinese-wage transfers")


Real wage / real expenditure (% change), transfers in US wage vs Chinese wage.
Sorted by y0: top = deficit countries (y0 < 1), bottom = surplus countries (y0 > 1)
         y0  transfer_share  real_wage_US  real_wage_CN  real_wage_diff  real_exp_US  real_exp_CN  real_exp_diff
ROW 0.95298         0.04702       0.01021       0.02200         0.01178      0.00944      0.11936        0.10992
GRC 0.95384         0.04616       0.00101       0.01531         0.01430      0.00079      0.11054        0.10975
BGR 0.97735         0.02265       0.00173       0.00779         0.00606      0.00149      0.05578        0.05429
PRT 0.98422         0.01578       0.00078       0.00643         0.00565      0.00065      0.03976        0.03911
USA 0.98472         0.01528       0.00182       0.00590         0.00408      0.00182      0.03761        0.03580
LVA 0.98703         0.01297       0.00133       0.00581         0.00449      0.00125      0.03362        0.03237
BRA 0.98866         0.01134       0.00257    

**Part v results: transfers denominated in Chinese wages**

When transfers are measured in Chinese wages, they become more valuable in US-wage terms, because the Chinese wage rises about 2.1% after the shock. What matters is each country's baseline transfer position, which the data gives through the income-to-spending ratio y0 from part ii. Net recipients (y0 < 1: 12 of the 40 non-China countries, for example Greece at 0.954 and the US at 0.985) receive more purchasing power and gain more, with real expenditure up to 0.11 percentage points higher than in part iii. Net payers (y0 > 1: the other 28, for example Germany at 1.058, Taiwan at 1.067, the Netherlands at 1.080 and Ireland at 1.112) pay more and gain less, with real expenditure up to 0.26 percentage points lower (Ireland goes from +0.004% to −0.257%).

The pattern is systematic: the sign of the change in real expenditure matches the sign of (1 − y0) in all 40 countries, and regressing the change on (1 − y0) gives a slope of 2.37 Real wages follow the same pattern with smaller changes (slope 0.195).

Compared with part iii, where all 40 countries gained, 28 countries now have negative real expenditure changes and 17 have negative real wages, so for most countries the choice of transfer numeraire matters more than the productivity shock itself.

In [58]:
# EXPORT RESULTS FOR EACH SECTION TO THE OUTPUT FOLDER

tables = {}

# ---- Part ii: calibration ---------------------------------------------------
# The full matrix X0. Rows are destinations n (buyers), columns are origins i
# (sellers), so entry [n, i] is what country n buys from country i.
tables["part2_X0_matrix"] = (
    pd.DataFrame(X0_ni, index=countries, columns=countries),
    "destination n (rows) / origin i (columns)",
    4,
)

# Spending, sales, and the income-to-spending ratio y0 for every country.
tables["part2_income_to_spending"] = (
    pd.DataFrame(
        {
            "total_spending_X0_n": X0_n,                 # row sums of X0
            "total_sales_Y0_n": Y0_n,                    # column sums of X0
            "y0_n": y0_n,                                # sales / spending
            "transfer_share_1_minus_y0": 1.0 - y0_n,     # >0 receives, <0 pays transfers
            "position": np.where(y0_n < 1, "deficit (receives transfers)",
                                 "surplus (pays transfers)"),
        },
        index=countries,
    ),
    "country",
    8,
)

# ---- Part iii: wage changes from the 10% Chinese productivity shock ---------
tables["part3_wage_changes"] = (
    results.rename(columns={"pct_change_wage": "pct_change_wage_vs_US"}),
    "country",
    8,
)

# The solver's error at every iteration: shows it converged and shrank each step.
tables["part3_convergence"] = (
    pd.DataFrame({"iteration": np.arange(len(history)), "max_abs_excess_demand": history}),
    None,
    None,      # no rounding: the errors are tiny (~1e-10), rounding would wipe them out
)

# ---- Part iv: real wages, real expenditure, and regressions -----------------
tables["part4_real_wage_expenditure"] = (report, "country", 8)
tables["part4_regressions"] = (reg_table, None, 6)

# ---- Part v: transfers denominated in the Chinese wage ----------------------
tables["part5_comparison"] = (comparison, "country", 8)
tables["part5_regressions"] = (pd.DataFrame(reg_rows), None, 6)

# ---- Settings and run summary ------------------------------------------------
# NOTE: kappa and tol below must match the values you passed to solve_wages
# (0.5 and 1e-10 in the notebook). Edit here if you changed them.
tables["settings_and_checks"] = (
    pd.DataFrame(
        [
            ("theta (trade elasticity)", THETA),
            ("shock: T_hat for China", 1.1),
            ("kappa (step size)", 0.5),
            ("tol (convergence tolerance)", 1e-10),
            ("part iii: iterations to converge", len(history) - 1),
            ("part iii: final max |Z|", history[-1]),
            ("part v: iterations to converge", len(history_cn) - 1),
            ("part v: final max |Z|", history_cn[-1]),
            ("China wage change vs US, transfers in US wage (%)", (w_hat_us[CHN] - 1) * 100),
            ("China wage change vs US, transfers in Chinese wage (%)", (w_hat_cn[CHN] - 1) * 100),
        ],
        columns=["item", "value"],
        dtype=object,      # keeps 257 as an integer and tiny errors unrounded
    ),
    None,
    None,
)


# WRITE THE CSV FILES
def rounded(df, decimals):
    """Round only the numeric columns (leave text columns like 'position' alone).
    decimals=None means leave the table exactly as it is."""
    if decimals is None:
        return df
    df = df.copy()
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].round(decimals)
    return df

saved = []
for name, (df, index_label, decimals) in tables.items():
    out = rounded(df, decimals)
    if index_label is None:
        out.to_csv(output_path / f"{name}.csv", index=False)
    else:
        out.to_csv(output_path / f"{name}.csv", index=True, index_label=index_label)
    saved.append(f"{name}.csv")


# WRITE ONE EXCEL WORKBOOK WITH A SHEET PER TABLE
# Convenient for markers: one file, one tab per table. Requires openpyxl.
try:
    with pd.ExcelWriter(output_path / "EK_model_results.xlsx") as writer:
        for name, (df, index_label, decimals) in tables.items():
            out = rounded(df, decimals)
            if index_label is None:
                out.to_excel(writer, sheet_name=name[:31], index=False)
            else:
                out.to_excel(writer, sheet_name=name[:31], index=True, index_label=index_label)
    saved.append("EK_model_results.xlsx")
except ImportError:
    print("openpyxl is not installed, so the Excel workbook was skipped "
          "(the CSV files were saved). To get it: pip install openpyxl")